In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, box

# Create points
points = gpd.GeoDataFrame(
    geometry=[
        Point(200, 200),
        Point(400, 600),
        Point(800, 800),
        Point(1200, 400)
    ],
    crs="EPSG:32643"
)

# Create buffers
buffer1 = points.copy()
buffer1["geometry"] = points.geometry.buffer(100)

buffer2 = points.copy()
buffer2["geometry"] = points.geometry.buffer(500)

buffer3 = points.copy()
buffer3["geometry"] = points.geometry.buffer(1000)

# Plot
ax = points.plot(color="yellow", edgecolor="black")
buffer3.plot(ax=ax, color="lightblue", edgecolor="blue", alpha=0.3)
buffer2.plot(ax=ax, color="lightgreen", edgecolor="green", alpha=0.4)
buffer1.plot(ax=ax, color="orange", edgecolor="red", alpha=0.6)

ax.set_title("Points with 100m, 500m and 1km Buffers")
ax.set_xlabel("X Coordinate (m)")
ax.set_ylabel("Y Coordinate (m)")
ax.grid(True)

plt.show()


In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, box

# Create overlapping rectangles
polygons = gpd.GeoDataFrame(
    geometry=[
        box(150, 150, 400, 400),
        box(350, 350, 600, 600)
    ],
    crs="EPSG:32643"
)
buffers = points.copy()
buffers["geometry"] = points.geometry.buffer(100)
# Overlay (intersection)
result = gpd.overlay(polygons, buffers, how="intersection")

# Plot
ax = buffers.plot(color="red", edgecolor="black")

polygons.plot(ax=ax, color="lightblue", alpha=0.5, edgecolor="blue")
result.plot(ax=ax, color="yellow", edgecolor="black")

ax.set_title("Polygon-Buffer Overlay (Intersection)")
ax.set_xlabel("X Coordinate (m)")
ax.set_ylabel("Y Coordinate (m)")
ax.grid(True)

plt.show()

In [ ]:
import geopandas as gpd
boundary = r"D:\Programming Lab\Lab-06\nedumangad\nedumangad_boundary.shp"
bp = r"D:\Programming Lab\Lab-06\nedumangad\buildings\nedumangad_buildings.shp"
roads = r"D:\Programming Lab\Lab-06\nedumangad\roads\nedumangad_road.shp"
water = r"D:\Programming Lab\Lab-06\nedumangad\waterways\nedumangad_waterways.shp"
pois = r"D:\Programming Lab\Lab-06\nedumangad\pois\nedumangad_pois.shp"
lulc = r"D:\Programming Lab\Lab-06\nedumangad\landuse\nedumangad_landuse.shp"

bry = gpd.read_file(boundary)
bry.columns

b = gpd.read_file(bp)
b.columns

r = gpd.read_file(roads)
r.columns

w = gpd.read_file(water)
w.columns

p = gpd.read_file(pois)
p.columns

l = gpd.read_file(lulc)
l.columns

bry = bry.set_crs("EPSG:4326", allow_override=True)
bry = bry.to_crs("EPSG:32643")

b = b.set_crs("EPSG:4326", allow_override=True)   # rewrite the metadata
b = b.to_crs("EPSG:32643")                        # actually transform it

r = r.set_crs("EPSG:4326", allow_override=True)
r = r.to_crs("EPSG:32643")

w = w.set_crs("EPSG:4326", allow_override=True)
w = w.to_crs("EPSG:32643")

p = p.set_crs("EPSG:4326", allow_override=True)
p = p.to_crs("EPSG:32643")

l = l.set_crs("EPSG:4326", allow_override=True)
l = l.to_crs("EPSG:32643")

# 1. Number of buildings
print("Total buildings:", len(b))

# 2. Primary road names
primary = r[
    r.astype(str).apply(
        lambda col: col.str.contains(
            "primary", case=False, na=False
        )
    ).any(axis=1)
]

print("Primary roads:")
if "name" in primary.columns:
    print(primary["name"].dropna().unique()[:10])
else:
    print(primary.drop(columns=primary.geometry.name).head(5))

# 3. Total waterways
print("Total waterways:", len(w))

# 4. Total points of interest
print("Total POIs:", len(p))

# 5. Area of Nedumangad
area_km2 = bry.geometry.union_all().area / 1_000_000
print("Area of Nedumangad:", round(area_km2, 2), "sq. km")

In [ ]:

# Select rivers and streams
rivers = w[w["fclass"].isin(["river", "stream"])]

# Create 200 m buffer
river_buffer = gpd.GeoDataFrame(
    geometry=[rivers.buffer(200).union_all()],
    crs="EPSG:32643"
)

# Find buildings within 200 m
e = gpd.sjoin(
    b, river_buffer,
    predicate="intersects",
    how="inner"
)

print("Total buildings:", len(b))
print("Buildings within 200 m of rivers and streams:", len(e))
print("Exposed share:", round(len(e) / len(b), 4))

# Print 10 building names
if "name" in e.columns:
    print("\n10 building names:")
    print(e["name"].dropna().drop_duplicates().head(10).to_string(index=False))
else:
    print("\nNo 'name' column found.")
    print("Available columns:", b.columns.tolist())
    print("\nSample building attributes:")
    print(e.drop(columns="geometry", errors="ignore").head(10))

# Plot results
fig, ax = plt.subplots(figsize=(9, 9))

river_buffer.plot(ax=ax, color="lightblue", alpha=0.5)
b.plot(ax=ax, color="lightgrey", edgecolor="none")
e.plot(ax=ax, color="red", edgecolor="red", markersize=8)
rivers.plot(ax=ax, color="blue", linewidth=1)

plt.title("Buildings within 200 m of Rivers and Streams")
plt.xlabel("Easting (m)")
plt.ylabel("Northing (m)")
plt.show()




In [ ]:

import geopandas as gpd
import matplotlib.pyplot as plt

# 1. Select primary and trunk roads
major_roads = r[r["fclass"].isin(["primary", "trunk"])]

# 2. Select hospitals from POIs
hospitals = p[p["fclass"].str.contains(
    "hospital", case=False, na=False
)]

# 3. Create a 750 m buffer around major roads
road_buffer = gpd.GeoDataFrame(
    geometry=[major_roads.buffer(750).union_all()],
    crs="EPSG:32643"
)

# 4. Find hospitals within 750 m of major roads
near_hospitals = gpd.sjoin(
    hospitals, road_buffer,
    predicate="intersects",
    how="inner"
)

near_hospitals = near_hospitals.loc[
    ~near_hospitals.index.duplicated()
]

print("Total hospitals:", len(hospitals))
print("Hospitals within 750 m of primary and trunk roads:",
      len(near_hospitals))

# 5. Print 10 hospital names
if "name" in near_hospitals.columns:
    print("\nHospital names:")
    print(
        near_hospitals["name"]
        .dropna()
        .drop_duplicates()
        .head(10)
        .to_string(index=False)
    )
else:
    print("\nNo 'name' column found.")
    print("Available columns:", hospitals.columns.tolist())

# 6. Plot the results
fig, ax = plt.subplots(figsize=(10, 10))

road_buffer.plot(
    ax=ax, color="lightblue", alpha=0.5
)

major_roads.plot(
    ax=ax, color="black", linewidth=0.8
)

hospitals.plot(
    ax=ax, color="lightgrey", markersize=15
)

near_hospitals.plot(
    ax=ax, color="red", markersize=35
)

plt.title("Hospitals within 750 m of Primary and Trunk Roads")
plt.xlabel("Easting (m)")
plt.ylabel("Northing (m)")
plt.legend(handles=[
    plt.Line2D([0], [0], color="black", label="Primary and trunk roads"),
    plt.Line2D([0], [0], color="lightblue", linewidth=8,
               label="750 m road buffer"),
    plt.Line2D([0], [0], marker="o", color="red", linestyle="",
               label="Nearby hospitals")
])
plt.tight_layout()
plt.show()


In [ ]:

import geopandas as gpd
import matplotlib.pyplot as plt

# Select bridges and rivers/canals
bridges = r[r["bridge"].astype(str).str.upper() == "T"]
rivers = w[w["fclass"].isin(["river", "canals"])]

# Find intersections
e = gpd.overlay(
    rivers,
    bridges,
    how="intersection",
    keep_geom_type=False
)

# Print results
print("Total bridges on top of rivers/canals:", len(e))

print("\nBridge names (any 10):")
if "name_2" in e.columns:
    print(e["name_2"].dropna().drop_duplicates().head(10).to_string(index=False))
else:
    print("Bridge name column not found. Available columns:", e.columns.tolist())

# Plot results
ax = rivers.plot(
    figsize=(9, 9),
    linewidth=2,
    alpha=0.6,
    color="blue",
    label="Rivers and canals"
)

bridges.plot(
    ax=ax,
    color="lightgrey",
    linewidth=1,
    label="Bridges"
)

e.plot(
    ax=ax,
    color="red",
    linewidth=3,
    label="Bridge-river intersections"
)

plt.title("Bridges on Rivers and Canals")
plt.legend()
plt.show()


In [ ]:
# 1. Select primary and tertiary roads
roads_selected = r[r["fclass"].isin(["primary", "tertiary"])]

# 2. Select forest areas from LULC
forest = l[l.astype(str).apply(
    lambda col: col.str.contains(
        "forest", case=False, na=False
    )
).any(axis=1)]


# 3. Find roads intersecting forests
forest_union = forest.geometry.union_all()

roads_in_forest = roads_selected[
    roads_selected.geometry.intersects(forest_union)
]

# 4. Print results
print("Total primary and tertiary roads inside forest:",
      len(roads_in_forest))

print("\nNames of roads inside forest:")
print(roads_in_forest["name"])
